In [1]:
from vizdoom import *
import cv2
import numpy as np
import os

In [2]:
# Gym imports
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Discrete, Box

In [3]:
# Stable Baselines3 imports
from stable_baselines3 import PPO
from stable_baselines3.common import env_checker
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor

In [4]:
# Paths
CONFIG_PATH = './github/ViZDoom/scenarios/take_cover.cfg'
CHECKPOINT_DIR = './train/train_take_cover'
LOG_DIR = './logs/log_take_cover'
MODEL_DIR = './models/take_cover_best'

In [5]:
# Create Vizdoom OpenAI Gym Environment
class VizDoomGym(Env): 
    def __init__(self, render=False): 
        # Setup the game 
        super().__init__()
        self.frame_skip = 4
        self.game = DoomGame()
        self.game.load_config(CONFIG_PATH)
        
        # Render frame logic
        self.game.set_window_visible(render)
        
        # Start the game 
        self.game.init()
        
        # Create the action space and observation space
        self.observation_space = Box(low=0, high=255, shape=(100,160,1), dtype=np.uint8) 
        self.action_space = Discrete(2) # Move left, move right
        self._actions = np.eye(2, dtype=np.uint8)
        
    # This is how we take a step in the environment
    def step(self, action):
        # Specify action and take step 
        reward = self.game.make_action(self._actions[action].tolist(), self.frame_skip) 
        
        # Get the new state of the game and check if it's done
        if self.game.get_state(): 
            obs    = self._process_frame(self.game.get_state().screen_buffer)
            health = self.game.get_state().game_variables[0]
            info   = {"health": health}
        else: 
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {"health": 0}
        
        terminated = self.game.is_episode_finished()
        truncated = False

        return obs, reward, terminated, truncated, info
    
    # Define how to render the game or environment 
    def render(self): 
        pass
    
    # Starting a new game 
    def reset(self, seed=None, options=None): 
        super().reset(seed=seed)
        self.game.new_episode()
        state = self.game.get_state()

        if state is not None:
            obs = self._process_frame(state.screen_buffer)
        else:
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)

        
        return obs, {}
    
    # Call to close down the game
    def close(self): 
        self.game.close()

    def _process_frame(self, buffer: np.ndarray):
        hwc  = np.moveaxis(buffer, 0, -1)                           
        gray = cv2.cvtColor(hwc, cv2.COLOR_RGB2GRAY)               
        resized = cv2.resize(gray, (160, 100), interpolation=cv2.INTER_CUBIC)
        clipped = np.clip(resized, 0, 255).astype(np.uint8)         
        return clipped.reshape(100, 160, 1)

In [6]:
# Creates the env wrapped with Monitor
def make_env(render = False):
    def _init():
        env = VizDoomGym(render=render)
        env = Monitor(env)
        return env
    return _init

In [7]:
# Sanity check 
env = VizDoomGym(render=False)
env_checker.check_env(env)
env.close()

In [8]:
# Build a vectorized environment for training
env = DummyVecEnv([make_env(render=False)])
env = VecFrameStack(env, n_stack=4, channels_order='last')

# Environment for a separate evaluation instance
eval_env = DummyVecEnv([make_env(render=False)])
eval_env = VecFrameStack(eval_env, n_stack=4, channels_order='last')

In [9]:
# Callbacks
# Saving the best model
eval_callback = EvalCallback(
        eval_env,
        best_model_save_path = MODEL_DIR,
        log_path             = LOG_DIR,
        eval_freq            = 10000,            
        n_eval_episodes      = 10,               
        deterministic        = True,
        render               = False,
        verbose              = 1,
)

#### Training the model

In [10]:
model = PPO(
    env = env,
    tensorboard_log=LOG_DIR,
    policy = 'CnnPolicy',
    learning_rate=3e-5,
    n_steps=4096,
    batch_size=64, 
    n_epochs=4,
    clip_range=0.1,
    ent_coef=0.01,
    verbose=1,
)

/home/acaia/AgenticTakeCover/AgenticTakeCover/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Using cpu device
Wrapping the env in a VecTransposeImage.


In [11]:
model.learn(total_timesteps=1000000, callback=eval_callback)

Logging to ./logs/log_take_cover/PPO_1


/home/acaia/AgenticTakeCover/AgenticTakeCover/.venv/lib/python3.10/site-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7f4f8ac85de0> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7f514867aad0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 77.9     |
|    ep_rew_mean     | 310      |
| time/              |          |
|    fps             | 164      |
|    iterations      | 1        |
|    time_elapsed    | 24       |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 78.3        |
|    ep_rew_mean          | 312         |
| time/                   |             |
|    fps                  | 98          |
|    iterations           | 2           |
|    time_elapsed         | 83          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.001600944 |
|    clip_fraction        | 0.0807      |
|    clip_range           | 0.1         |
|    entropy_loss         | -0.692      |
|    explained_variance   | -0.00112    |
|    learning_rate        | 3e

In [12]:
env.close()
eval_env.close()